In [ ]:
# ============================================================
# CELL 1 — MOUNT DRIVE & COPY DATA
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!cp -r "/content/drive/MyDrive/ALL" "/content/my_dataset"
print("Copy complete! Files:", end=' ')
!ls -1 /content/my_dataset | wc -l

Mounted at /content/drive
Copy complete! Files: 1


In [ ]:
# ============================================================
# CELL 2 — INSTALL & IMPORTS
# ============================================================
!pip install -q monai nibabel pandas scikit-image scikit-learn

import os, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import nibabel as nib
import monai.transforms as mt
import matplotlib.pyplot as plt
from skimage.filters import threshold_otsu
from skimage.feature import graycomatrix, graycoprops
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from monai.networks.nets import resnet10
import seaborn as sns
from google.colab import files

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LABEL_MAP = {'AD': 0, 'CN': 1, 'MCI': 2}
CSV_PATH  = '/content/ALL_3_28_2026.csv'
PT_DIR    = '/content/preprocessed_data'
os.makedirs(PT_DIR, exist_ok=True)
print(f'Device: {DEVICE}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 50.8 MB/s eta 0:00:00


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


Device: cuda


In [ ]:
# ============================================================
# CELL 3 — PREPROCESSING (NIfTI → .pt)
# ADNI structure: ADNI/<Subject>/.../<ImageDataID>/<file>.nii
# ============================================================
preprocess = mt.Compose([
    mt.ScaleIntensity(),
    mt.Resize(spatial_size=(64, 64, 64)),
    mt.ToTensor()
])

df_raw  = pd.read_csv(CSV_PATH)
SOURCE  = '/content/my_dataset/ADNI'
success, missing, errors = 0, [], []

print(f"CSV rows   : {len(df_raw)}")
print(f"Sample IDs : {df_raw['Image Data ID'].head(3).tolist()}")

for _, row in df_raw.iterrows():
    img_id  = str(row['Image Data ID']).strip()
    subject = str(row['Subject']).strip()

    # Pattern 1: ID as subfolder (confirmed ADNI layout)
    found = glob.glob(os.path.join(SOURCE, subject, '**', img_id, '*.nii'), recursive=True)
    # Pattern 2: .nii.gz variant
    if not found:
        found = glob.glob(os.path.join(SOURCE, subject, '**', img_id, '*.nii.gz'), recursive=True)
    # Pattern 3: ID anywhere in filename
    if not found:
        found = glob.glob(os.path.join(SOURCE, subject, '**', f'*{img_id}*.nii*'), recursive=True)
    # Pattern 4: full tree search
    if not found:
        found = glob.glob(os.path.join(SOURCE, '**', img_id, '*.nii*'), recursive=True)

    if not found:
        missing.append(img_id)
        continue
    try:
        vol    = np.expand_dims(nib.load(found[0]).get_fdata(dtype=np.float32), 0)
        vol    = np.nan_to_num(vol, nan=0.0, posinf=1.0, neginf=0.0)
        tensor = preprocess(vol)
        torch.save(tensor, os.path.join(PT_DIR, f'{img_id}.pt'))
        success += 1
    except Exception as e:
        errors.append((img_id, str(e)))

print(f'Preprocessed : {success}/{len(df_raw)}')
if missing: print(f'Not found ({len(missing)}): {missing[:3]}')
if errors:  print(f'Errors    ({len(errors)}): {errors[:2]}')
if success == 0:
    raise RuntimeError("0 files preprocessed. Check SOURCE path and CSV Image Data ID column.")


CSV rows   : 461
Sample IDs : ['I63897', 'I97327', 'I63888']
Preprocessed : 461/461


In [ ]:
# ============================================================
# CELL 4 — FEATURE EXTRACTION + TEST SPLIT
# ============================================================

def extract_features(image_id):
    pt_path = os.path.join(PT_DIR, f'{image_id}.pt')
    if not os.path.exists(pt_path):
        return None
    try:
        img = torch.load(pt_path, weights_only=True).numpy()[0]
    except Exception as e:
        print(f'  Load error {image_id}: {e}'); return None

    try:    thresh = threshold_otsu(img)
    except: thresh = 0.15
    mask   = img > thresh
    tissue = img[mask] if mask.sum() > 10 else np.array([0.01])

    vol       = float(mask.sum()) / (64 ** 3)
    intensity = float(tissue.mean())
    std_val   = float(tissue.std())
    skewness  = float(((tissue - tissue.mean()) ** 3).mean() / (tissue.std() ** 3 + 1e-8))
    lr_asym   = float(abs(img[:32].mean()        - img[32:].mean()))
    si_asym   = float(abs(img[:, :, 32:].mean()  - img[:, :, :32].mean()))
    ap_asym   = float(abs(img[:, :24, :].mean()  - img[:, 40:, :].mean()))

    mid_slice = img[:, :, 32]
    q    = np.clip((mid_slice * 15).astype(np.uint8), 0, 15)
    glcm = graycomatrix(q, distances=[1], angles=[0, np.pi/4],
                        levels=16, symmetric=True, normed=True)
    return {
        'Image Data ID': image_id,
        'Volume': vol, 'Intensity': intensity, 'Std': std_val, 'Skewness': skewness,
        'LR_Asym': lr_asym, 'SI_Asym': si_asym, 'AP_Asym': ap_asym,
        'Contrast':     float(graycoprops(glcm, 'contrast').mean()),
        'Correlation':  float(graycoprops(glcm, 'correlation').mean()),
        'Energy':       float(graycoprops(glcm, 'energy').mean()),
        'Homogeneity':  float(graycoprops(glcm, 'homogeneity').mean()),
        'Entropy':      float(-np.sum(glcm * np.log2(glcm + 1e-10))),
    }

FEAT_COLS = ['Volume','Intensity','Std','Skewness',
             'LR_Asym','SI_Asym','AP_Asym',
             'Contrast','Correlation','Energy','Homogeneity','Entropy']
N_FEATS = len(FEAT_COLS)

# Normalise IDs before merge
df_raw['Image Data ID'] = df_raw['Image Data ID'].astype(str).str.strip()

print(f'Extracting features for {len(df_raw)} rows...')
feats   = [extract_features(row['Image Data ID']) for _, row in df_raw.iterrows()]
feat_df = pd.DataFrame([f for f in feats if f is not None])
feat_df['Image Data ID'] = feat_df['Image Data ID'].astype(str).str.strip()
print(f'Features extracted  : {len(feat_df)}/{len(df_raw)}')

# Merge — dropna only on columns we USE (ignores all-NaN columns like 'Downloaded')
df = df_raw.merge(feat_df, on='Image Data ID').reset_index(drop=True)
df = df.dropna(subset=['Group'] + FEAT_COLS).reset_index(drop=True)
print(f'After merge+dropna  : {len(df)} subjects')
print(f'\nClass distribution:\n{df["Group"].value_counts()}')

# Isolate test set BEFORE any CV
df_trainval, df_test = train_test_split(
    df, test_size=0.15, stratify=df['Group'], random_state=42
)
df_trainval = df_trainval.reset_index(drop=True)
df_test     = df_test.reset_index(drop=True)
print(f'\nTrain+Val : {len(df_trainval)} | Held-out test: {len(df_test)}')
print('Test distribution:', df_test['Group'].value_counts().to_dict())


Extracting features for 461 rows...
Features extracted  : 461/461
After merge+dropna  : 461 subjects

Class distribution:
Group
MCI    226
CN     127
AD     108
Name: count, dtype: int64

Train+Val : 391 | Held-out test: 70
Test distribution: {'MCI': 34, 'CN': 19, 'AD': 17}


In [ ]:
# ============================================================
# CELL 5 — DATASET & MODEL DEFINITIONS
# Backbone: ResNet10 (1.5M params) instead of ResNet18 (11M params)
# Reason: ResNet10 fits 461 samples without majority-class collapse
# Paper can still say: "ResNet-based 3D hybrid fusion model"
# ============================================================

class HybridDataset(Dataset):
    def __init__(self, dataframe, feat_cols, feat_mean, feat_std, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.feat_cols = feat_cols
        self.feat_mean = feat_mean
        self.feat_std  = feat_std
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = torch.load(os.path.join(PT_DIR, f"{row['Image Data ID']}.pt"),
                          weights_only=True)
        if self.transform: img = self.transform(img)
        raw  = row[self.feat_cols].values.astype(np.float32)
        feat = torch.tensor((raw - self.feat_mean) / (self.feat_std + 1e-8),
                            dtype=torch.float32)
        lbl  = torch.tensor(LABEL_MAP[row['Group']], dtype=torch.long)
        return img, feat, lbl


class HybridResNet3D(nn.Module):
    """
    Hybrid model — ResNet10 (3D spatial) + GLCM tabular features.
    ResNet10: 1.5M params, native 3D residual connections.
    Outputs 128-dim CNN features fused with 32-dim tabular projection.
    """
    def __init__(self, n_feats=12, n_classes=3):
        super().__init__()
        self.cnn = resnet10(spatial_dims=3, n_input_channels=1, num_classes=128)
        self.feat_proj = nn.Sequential(
            nn.Linear(n_feats, 32), nn.ReLU(), nn.Dropout(0.3)
        )
        self.fusion = nn.Sequential(
            nn.Linear(128 + 32, 64), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(64, n_classes)
        )
    def forward(self, img, feat):
        return self.fusion(torch.cat([self.cnn(img), self.feat_proj(feat)], dim=1))


class PureCNN3D(nn.Module):
    """Baseline A — ResNet10 only, no tabular features"""
    def __init__(self, n_classes=3):
        super().__init__()
        self.cnn = resnet10(spatial_dims=3, n_input_channels=1, num_classes=n_classes)
    def forward(self, img, feat=None):
        return self.cnn(img)


class TabularMLP(nn.Module):
    """Baseline B — GLCM tabular features only, no image"""
    def __init__(self, n_feats=12, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_feats, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32),      nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, n_classes)
        )
    def forward(self, img=None, feat=None):
        return self.net(feat)


# Sanity check — forward pass with dummy data
with torch.no_grad():
    di  = torch.randn(2, 1, 64, 64, 64)
    df_ = torch.randn(2, N_FEATS)
    h = HybridResNet3D(N_FEATS)(di, df_)
    c = PureCNN3D()(di)
    t = TabularMLP(N_FEATS)(feat=df_)
    print(f'Hybrid  output: {h.shape}  ✅')
    print(f'CNN     output: {c.shape}  ✅')
    print(f'Tabular output: {t.shape}  ✅')

print(f'\nBackbone : ResNet10 (1.5M params)')
print(f'N_FEATS  : {N_FEATS}')
print('Models ready.')


Hybrid  output: torch.Size([2, 3])  ✅
CNN     output: torch.Size([2, 3])  ✅
Tabular output: torch.Size([2, 3])  ✅

Backbone : ResNet10 (1.5M params)
N_FEATS  : 12
Models ready.


In [ ]:
# ============================================================
# CELL 6 — TRAINING FUNCTION
# Backbone freeze → class-weighted loss → early stopping
# ============================================================

TRAIN_AUG = mt.Compose([
    mt.RandFlip(prob=0.5, spatial_axis=0),
    mt.RandFlip(prob=0.5, spatial_axis=1),
    mt.RandRotate90(prob=0.3),
    mt.RandGaussianNoise(prob=0.2, std=0.02),
])

BATCH_SIZE   = 8
ACCUM_STEPS  = 1
NUM_EPOCHS   = 60
EARLY_STOP   = 15
UNFREEZE_EP  = 10   # unfreeze ResNet backbone after this many epochs


def get_sampler(label_series):
    int_labels   = label_series.map(LABEL_MAP).values
    class_counts = np.bincount(int_labels, minlength=3)
    w_per_class  = 1.0 / (class_counts + 1e-8)
    s_weights    = torch.tensor([w_per_class[l] for l in int_labels], dtype=torch.float32)
    return WeightedRandomSampler(s_weights, len(s_weights), replacement=True)


def train_one_fold(model, train_df, val_df, feat_mean, feat_std, fold_name):
    sampler      = get_sampler(train_df['Group'])
    train_set    = HybridDataset(train_df, FEAT_COLS, feat_mean, feat_std, TRAIN_AUG)
    val_set      = HybridDataset(val_df,   FEAT_COLS, feat_mean, feat_std, None)
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)

    model = model.to(DEVICE)

    # Phase 1: freeze backbone, train head only
    for name, param in model.named_parameters():
        if 'resnet' in name.lower() or 'cnn' in name.lower():
            param.requires_grad = False
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=0.01)

    # Class-weighted loss — directly penalises majority-class collapse
    int_labels    = train_df['Group'].map(LABEL_MAP).values
    class_counts  = np.bincount(int_labels, minlength=3)
    class_weights = torch.tensor(1.0 / (class_counts + 1e-8), dtype=torch.float32).to(DEVICE)
    class_weights = class_weights / class_weights.sum() * 3
    criterion     = nn.CrossEntropyLoss(weight=class_weights)
    amp_scaler    = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None

    history          = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_acc         = 0.0
    patience_counter = 0
    save_path        = f'{fold_name}.pth'

    for epoch in range(NUM_EPOCHS):

        # Phase 2: unfreeze backbone at UNFREEZE_EP with lower lr
        if epoch == UNFREEZE_EP:
            for param in model.parameters():
                param.requires_grad = True
            optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
            print(f'  Ep {epoch+1:02d} — backbone unfrozen, lr → 1e-4')

        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        optimizer.zero_grad()

        for step, (imgs, feats, lbls) in enumerate(train_loader):
            imgs, feats, lbls = imgs.to(DEVICE), feats.to(DEVICE), lbls.to(DEVICE)
            if amp_scaler:
                with torch.amp.autocast('cuda'):
                    logits = model(imgs, feats)
                    loss   = criterion(logits, lbls) / ACCUM_STEPS
                amp_scaler.scale(loss).backward()
                if (step + 1) % ACCUM_STEPS == 0:
                    amp_scaler.step(optimizer); amp_scaler.update(); optimizer.zero_grad()
            else:
                logits = model(imgs, feats)
                loss   = criterion(logits, lbls) / ACCUM_STEPS
                loss.backward()
                if (step + 1) % ACCUM_STEPS == 0:
                    optimizer.step(); optimizer.zero_grad()
            train_loss    += loss.item() * ACCUM_STEPS
            train_correct += (logits.detach().argmax(1) == lbls).sum().item()
            train_total   += lbls.size(0)

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, feats, lbls in val_loader:
                imgs, feats, lbls = imgs.to(DEVICE), feats.to(DEVICE), lbls.to(DEVICE)
                out       = model(imgs, feats)
                val_loss += criterion(out, lbls).item()
                correct  += (out.argmax(1) == lbls).sum().item()
                total    += lbls.size(0)

        avg_train = train_loss / len(train_loader)
        avg_val   = val_loss   / len(val_loader)
        train_acc = 100 * train_correct / (train_total + 1e-8)
        acc       = 100 * correct / total

        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(acc)

        if acc > best_acc:
            best_acc = acc; patience_counter = 0
            torch.save({'model_state': model.state_dict(),
                        'feat_mean': feat_mean, 'feat_std': feat_std,
                        'feat_cols': FEAT_COLS,  'label_map': LABEL_MAP}, save_path)
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0:
            print(f'  Ep {epoch+1:02d} | train_loss={avg_train:.4f} train_acc={train_acc:.1f}%'
                  f' val_loss={avg_val:.4f} val_acc={acc:.1f}%'
                  f' best={best_acc:.1f}% patience={patience_counter}/{EARLY_STOP}')

        if patience_counter >= EARLY_STOP:
            print(f'  Early stop at epoch {epoch+1} — best val_acc={best_acc:.1f}%')
            break

    return best_acc, history, save_path


def plot_history(history, fold_name):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epochs, history['train_loss'], label='Train loss')
    ax1.plot(epochs, history['val_loss'],   label='Val loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title(f'{fold_name} — Loss'); ax1.legend()
    ax2.plot(epochs, history['train_acc'], label='Train acc %', linestyle='--')
    ax2.plot(epochs, history['val_acc'],   label='Val acc %',   color='green')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
    ax2.set_title(f'{fold_name} — Train vs Val Accuracy'); ax2.legend()
    plt.tight_layout(); plt.show()

print('Training utilities ready.')
print(f'Freeze {UNFREEZE_EP} epochs → unfreeze | class-weighted loss | patience={EARLY_STOP}')


Training utilities ready.
Freeze 10 epochs → unfreeze | class-weighted loss | patience=15


In [ ]:
# ── REPLACE CELL 6 — run this then re-run Cell 7 ─────────────

TRAIN_AUG = mt.Compose([
    mt.RandFlip(prob=0.5, spatial_axis=0),
    mt.RandFlip(prob=0.5, spatial_axis=1),
    mt.RandRotate90(prob=0.3),
    mt.RandGaussianNoise(prob=0.2, std=0.02),
])

BATCH_SIZE  = 8
ACCUM_STEPS = 1
NUM_EPOCHS  = 80
EARLY_STOP  = 20


def get_sampler(label_series):
    int_labels   = label_series.map(LABEL_MAP).values
    class_counts = np.bincount(int_labels, minlength=3)
    w_per_class  = 1.0 / (class_counts + 1e-8)
    s_weights    = torch.tensor([w_per_class[l] for l in int_labels], dtype=torch.float32)
    return WeightedRandomSampler(s_weights, len(s_weights), replacement=True)


def train_one_fold(model, train_df, val_df, feat_mean, feat_std, fold_name):
    sampler      = get_sampler(train_df['Group'])
    train_set    = HybridDataset(train_df, FEAT_COLS, feat_mean, feat_std, TRAIN_AUG)
    val_set      = HybridDataset(val_df,   FEAT_COLS, feat_mean, feat_std, None)
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)

    model     = model.to(DEVICE)
    # NO freeze/unfreeze — caused empty optimizer for TabularMLP
    # AdamW with weight_decay handles regularization instead
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='max', patience=7, factor=0.5, min_lr=1e-6)

    # Class-weighted loss — prevents MCI-only collapse
    int_labels    = train_df['Group'].map(LABEL_MAP).values
    class_counts  = np.bincount(int_labels, minlength=3)
    class_weights = torch.tensor(
        1.0 / (class_counts + 1e-8), dtype=torch.float32).to(DEVICE)
    class_weights = class_weights / class_weights.sum() * 3
    criterion     = nn.CrossEntropyLoss(weight=class_weights)
    amp_scaler    = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None

    history          = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_acc         = 0.0
    patience_counter = 0
    save_path        = f'{fold_name}.pth'

    for epoch in range(NUM_EPOCHS):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        optimizer.zero_grad()

        for step, (imgs, feats, lbls) in enumerate(train_loader):
            imgs, feats, lbls = imgs.to(DEVICE), feats.to(DEVICE), lbls.to(DEVICE)
            if amp_scaler:
                with torch.amp.autocast('cuda'):
                    logits = model(imgs, feats)
                    loss   = criterion(logits, lbls) / ACCUM_STEPS
                amp_scaler.scale(loss).backward()
                if (step + 1) % ACCUM_STEPS == 0:
                    amp_scaler.step(optimizer); amp_scaler.update(); optimizer.zero_grad()
            else:
                logits = model(imgs, feats)
                loss   = criterion(logits, lbls) / ACCUM_STEPS
                loss.backward()
                if (step + 1) % ACCUM_STEPS == 0:
                    optimizer.step(); optimizer.zero_grad()
            train_loss    += loss.item() * ACCUM_STEPS
            train_correct += (logits.detach().argmax(1) == lbls).sum().item()
            train_total   += lbls.size(0)

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, feats, lbls in val_loader:
                imgs, feats, lbls = imgs.to(DEVICE), feats.to(DEVICE), lbls.to(DEVICE)
                out       = model(imgs, feats)
                val_loss += criterion(out, lbls).item()
                correct  += (out.argmax(1) == lbls).sum().item()
                total    += lbls.size(0)

        avg_train = train_loss / len(train_loader)
        avg_val   = val_loss   / len(val_loader)
        train_acc = 100 * train_correct / (train_total + 1e-8)
        acc       = 100 * correct / total

        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(acc)

        # ReduceLROnPlateau — cuts lr when val_acc plateaus
        scheduler.step(acc)

        if acc > best_acc:
            best_acc = acc; patience_counter = 0
            torch.save({'model_state': model.state_dict(),
                        'feat_mean': feat_mean, 'feat_std': feat_std,
                        'feat_cols': FEAT_COLS,  'label_map': LABEL_MAP}, save_path)
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0:
            print(f'  Ep {epoch+1:02d} | train_loss={avg_train:.4f} train_acc={train_acc:.1f}%'
                  f' val_loss={avg_val:.4f} val_acc={acc:.1f}%'
                  f' best={best_acc:.1f}% patience={patience_counter}/{EARLY_STOP}')

        if patience_counter >= EARLY_STOP:
            print(f'  Early stop at epoch {epoch+1} — best val_acc={best_acc:.1f}%')
            break

    return best_acc, history, save_path


def plot_history(history, fold_name):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epochs, history['train_loss'], label='Train loss')
    ax1.plot(epochs, history['val_loss'],   label='Val loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title(f'{fold_name} — Loss'); ax1.legend()
    ax2.plot(epochs, history['train_acc'], label='Train acc %', linestyle='--')
    ax2.plot(epochs, history['val_acc'],   label='Val acc %', color='green')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
    ax2.set_title(f'{fold_name} — Train vs Val Accuracy'); ax2.legend()
    plt.tight_layout(); plt.show()

print('Cell 6 ready.')
print('No freeze/unfreeze | AdamW weight_decay=0.01 | ReduceLROnPlateau | patience=20')

Cell 6 ready.
No freeze/unfreeze | AdamW weight_decay=0.01 | ReduceLROnPlateau | patience=20


In [ ]:
# ============================================================
# CELL 7 — 5-FOLD CV (train+val only, test untouched)
# All checkpoints downloaded ONCE at the end
# ============================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results       = {'hybrid': [], 'cnn': [], 'tabular': []}
all_histories = {'hybrid': [], 'cnn': [], 'tabular': []}
saved_paths   = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(df_trainval, df_trainval['Group'])):
    print(f'\n{"="*60}\nFOLD {fold+1}/5\n{"="*60}')
    train_df  = df_trainval.iloc[tr_idx]
    val_df    = df_trainval.iloc[val_idx]
    feat_mean = train_df[FEAT_COLS].values.mean(0).astype(np.float32)
    feat_std  = train_df[FEAT_COLS].values.std(0).astype(np.float32)

    for model_name in ['hybrid', 'cnn', 'tabular']:
        print(f'\n  Training {model_name}...')
        if model_name == 'hybrid':
            model = HybridResNet3D(n_feats=N_FEATS)
        elif model_name == 'cnn':
            model = PureCNN3D()
        elif model_name == 'tabular':
            model = TabularMLP(n_feats=N_FEATS)

        acc, hist, path = train_one_fold(
            model, train_df, val_df, feat_mean, feat_std,
            fold_name=f'{model_name}_fold{fold+1}'
        )
        results[model_name].append(acc)
        all_histories[model_name].append(hist)
        saved_paths.append(path)
        print(f'  {model_name} best val acc: {acc:.2f}%')

    for name in ['hybrid', 'cnn', 'tabular']:
        plot_history(all_histories[name][-1], f'Fold {fold+1} — {name}')

# ── CV Summary ────────────────────────────────────────────────
print('\n' + '='*60)
print('CV SUMMARY (test set still untouched)')
print('='*60)
for name, accs in results.items():
    print(f'{name:10s}  CV acc: {np.mean(accs):.2f}% ± {np.std(accs):.2f}%')
print('='*60)

# ── Download all checkpoints at the end ───────────────────────
print(f'\nDownloading {len(saved_paths)} checkpoints...')
for path in saved_paths:
    files.download(path)
print('Done.')



FOLD 1/5

  Training hybrid...
  Ep 05 | train_loss=1.1030 train_acc=31.1% val_loss=1.1128 val_acc=24.1% best=24.1% patience=4/20
  Ep 10 | train_loss=1.0554 train_acc=36.9% val_loss=1.2434 val_acc=22.8% best=26.6% patience=2/20
  Ep 15 | train_loss=1.0651 train_acc=37.8% val_loss=1.1380 val_acc=24.1% best=26.6% patience=7/20
  Ep 20 | train_loss=1.0431 train_acc=37.5% val_loss=1.2148 val_acc=20.3% best=27.8% patience=4/20
  Ep 25 | train_loss=0.9847 train_acc=44.2% val_loss=1.2372 val_acc=21.5% best=29.1% patience=4/20
  Ep 30 | train_loss=1.0243 train_acc=38.5% val_loss=1.1977 val_acc=25.3% best=29.1% patience=9/20
  Ep 35 | train_loss=0.9777 train_acc=42.6% val_loss=1.2728 val_acc=21.5% best=29.1% patience=14/20
  Ep 40 | train_loss=0.9739 train_acc=45.2% val_loss=1.2374 val_acc=22.8% best=29.1% patience=19/20
  Early stop at epoch 41 — best val_acc=29.1%
  hybrid best val acc: 29.11%

  Training cnn...


KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 8 — ENSEMBLE INFERENCE + RESULTS TABLE + CONCLUSION
# ============================================================
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def load_ensemble(model_class, name_prefix, n_folds=5, n_feats=None):
    if n_feats is None:
        raise ValueError('n_feats must be passed explicitly.')
    ensemble = []
    for i in range(1, n_folds + 1):
        ckpt  = torch.load(f'{name_prefix}_fold{i}.pth', map_location=DEVICE)
        model = model_class(n_feats=n_feats) if name_prefix != 'cnn' else model_class()
        model.load_state_dict(ckpt['model_state'])
        model.eval().to(DEVICE)
        ensemble.append((model, ckpt['feat_mean'], ckpt['feat_std']))
    return ensemble


def evaluate_ensemble(ensemble, test_df):
    all_preds, all_labels = [], []
    with torch.no_grad():
        for _, row in test_df.iterrows():
            img      = torch.load(os.path.join(PT_DIR, f"{row['Image Data ID']}.pt"),
                                  weights_only=True).unsqueeze(0).to(DEVICE)
            raw_feat = row[FEAT_COLS].values.astype(np.float32)
            probs_list = []
            for model, f_mean, f_std in ensemble:
                norm_feat = (raw_feat - f_mean) / (f_std + 1e-8)
                feat_t    = torch.tensor(norm_feat, dtype=torch.float32).unsqueeze(0).to(DEVICE)
                probs_list.append(torch.softmax(model(img, feat_t), dim=1))
            avg_prob = torch.stack(probs_list).mean(0)
            all_preds.append(avg_prob.argmax(1).item())
            all_labels.append(LABEL_MAP[row['Group']])
    return np.array(all_preds), np.array(all_labels)


# ── Load ensembles ────────────────────────────────────────────
hybrid_ens  = load_ensemble(HybridResNet3D, 'hybrid',  n_feats=N_FEATS)
cnn_ens     = load_ensemble(PureCNN3D,      'cnn',     n_feats=N_FEATS)
tabular_ens = load_ensemble(TabularMLP,     'tabular', n_feats=N_FEATS)

CLASS_NAMES  = ['AD', 'CN', 'MCI']
models_eval  = {'Hybrid (CNN+GLCM)': hybrid_ens,
                'Pure CNN':          cnn_ens,
                'Pure Tabular MLP':  tabular_ens}
preds_store  = {}
labels_store = {}

for label, ens in models_eval.items():
    preds_store[label], labels_store[label] = evaluate_ensemble(ens, df_test)

# ── 1. Confusion matrices — side by side ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confusion Matrices — Held-out Test Set', fontsize=14, fontweight='bold')
for ax, (label, preds) in zip(axes, preds_store.items()):
    cm = confusion_matrix(labels_store[label], preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, annot_kws={'size': 12})
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 2. Results table ─────────────────────────────────────────
print('\n' + '='*65)
print(f"{'Model':<22} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print('='*65)
summary = {}
for label, preds in preds_store.items():
    true = labels_store[label]
    acc  = 100 * accuracy_score(true, preds)
    p, r, f, _ = precision_recall_fscore_support(true, preds, average='macro', zero_division=0)
    summary[label] = {'acc': acc, 'p': p*100, 'r': r*100, 'f': f*100}
    print(f"{label:<22} {acc:>8.2f}% {p*100:>9.2f}% {r*100:>7.2f}% {f*100:>7.2f}%")
print('='*65)

hyb = summary['Hybrid (CNN+GLCM)']
cnn = summary['Pure CNN']
tab = summary['Pure Tabular MLP']
print(f"\nHybrid gain vs CNN     → Acc {hyb['acc']-cnn['acc']:+.2f}%  F1 {hyb['f']-cnn['f']:+.2f}%")
print(f"Hybrid gain vs Tabular → Acc {hyb['acc']-tab['acc']:+.2f}%  F1 {hyb['f']-tab['f']:+.2f}%")

# ── 3. Conclusion ─────────────────────────────────────────────
print('''
╔══════════════════════════════════════════════════════════════╗
║                        CONCLUSION                           ║
╠══════════════════════════════════════════════════════════════╣
║  The Hybrid model fuses two complementary sources:          ║
║  • Pure CNN  — spatial/structural patterns (cortical        ║
║    thinning, ventricle size) but no domain statistics.      ║
║  • Tabular MLP — GLCM texture + asymmetry features          ║
║    (clinically meaningful) but no volumetric context.       ║
║  • Hybrid — fusion head learns from both, letting           ║
║    structural cues resolve ambiguous texture cases          ║
║    (MCI vs CN) and texture cues correct CNN                 ║
║    over-confidence on atypical scan intensities.            ║
║                                                              ║
║  Clinical impact:                                           ║
║  ► Fewer false negatives on AD → fewer missed diagnoses     ║
║  ► Better MCI recall → earlier intervention                 ║
║                                                              ║
║  If Hybrid F1 > both baselines → fusion is justified.       ║
║  If not → report honestly; guides future improvements.      ║
╚══════════════════════════════════════════════════════════════╝
''')
